## Getting to know the data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
tweet = pd.read_csv('data/train.csv')
tweet.head()

In [ ]:
tweet.dtypes

In [ ]:
tweet.shape

In [ ]:
tweet["location"].head(5)
tweet["keyword"].head(5)
tweet["text"].head(5)

In [ ]:
nan_counts = tweet.isna().sum()
plot = nan_counts.plot.bar(title='Number of NaN values per column')
plt.xlabel('Columns')
plt.ylabel('Number of NaN values')
plt.show()

In [ ]:
tweet["keyword"].value_counts()

In [ ]:
# text length
tweet['text_length'] = tweet['text'].apply(len)
tweet['text_length'].head()

In [ ]:
# correlaion between text length and target
correlation = tweet['text_length'].corr(tweet['target'])
print(f'Correlation between text length and target: {correlation}')
# No correlation found

In [ ]:
# check presence of keywords in text
tweet['keyword_in_text'] = tweet.apply(lambda row: row['keyword'] in row['text'] if pd.notna(row['keyword']) else 0, axis=1)
tweet[['keyword', 'text', 'keyword_in_text']].head(100)

In [ ]:
# common disaster keywords in non-disaster tweets
non_disaster_tweets = tweet[tweet['target'] == 0]
common_keywords_non_disaster = non_disaster_tweets['keyword'].value_counts()
print("Common disaster keywords in non-disaster tweets:")
print(common_keywords_non_disaster)

In [ ]:
# common disaster keywords in disaster tweets
disaster_tweets = tweet[tweet['target'] == 1]
common_keywords_disaster = disaster_tweets['keyword'].value_counts()
print("Common disaster keywords in disaster tweets:")
print(common_keywords_disaster)

## Text Processing

In [ ]:
# Clean tweets (text) and keywords by removing any urls, mentions, hashtags, non-alphabetic chars and even stopwords and convert everything to lowercase
import string
tweet['text'] = tweet['text'].str.replace(r'http\S+|www\S+|https\S+', '', case=False)  
tweet['text'] = tweet['text'].str.replace(r'@\w+', '', case=False) 
tweet['text'] = tweet['text'].str.replace(r'#\w+', '', case=False)  
tweet['text'] = tweet['text'].str.replace(r'[^a-zA-Z\s]', '', case=False) 
tweet['text'] = tweet['text'].str.replace(r'\s+', ' ', case=False) 
tweet['text'] = tweet['text'].str.replace('#', '', case=False)
tweet['text'] = tweet['text'].str.replace("'", '', case=False)
tweet['text'] = tweet['text'].str.translate(str.maketrans('', '', string.punctuation))
tweet['text'] = tweet['text'].str.strip() 
tweet['text'] = tweet['text'].str.lower()  
tweet['text'].head(5)

In [ ]:
tweet.head()

In [ ]:
# let's do the same for keyword
tweet['keyword'] = tweet['keyword'].str.replace(r'http\S+|www\S+|https\S+', '', case=False)  
tweet['keyword'] = tweet['keyword'].str.replace(r'@\w+', '', case=False) 
tweet['keyword'] = tweet['keyword'].str.replace(r'#\w+', '', case=False)  
tweet['keyword'] = tweet['keyword'].str.replace(r'[^a-zA-Z\s]', '', case=False) 
tweet['keyword'] = tweet['keyword'].str.replace(r'\s+', ' ', case=False) 
tweet['keyword'] = tweet['keyword'].str.replace('#', '', case=False)
#remove numbers too
tweet['keyword'] = tweet['keyword'].str.replace(r'\d+', '', case=False)
tweet['keyword'] = tweet['keyword'].str.replace("'", '', case=False)
tweet['keyword'] = tweet['keyword'].str.translate(str.maketrans('', '', string.punctuation))
tweet['keyword'] = tweet['keyword'].str.strip() 
tweet['keyword'] = tweet['keyword'].str.lower()  
tweet['keyword'].head(20)
# check with non null values
tweet[tweet['keyword'].notnull()]['keyword'].head(30)

In [ ]:
# tweet["text"].value_counts()
print(tweet["keyword"].value_counts())

In [ ]:
# remove numbers from keywords as well without regex
for i in range(10):
    tweet['keyword'] = tweet['keyword'].str.replace(str(i), ' ')
print(tweet["keyword"].value_counts())

## Feature Engineering

#### Creation of vectorized text features? TF-IDF

As a way to practice my knwoledge, i'll go through BOW (Bag Of Words), TF-IDF, and Word Embeddings.

##### Bag of Words

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
bow_vectorizerText = CountVectorizer(stop_words='english')
bow_matrixText = bow_vectorizerText.fit_transform(tweet['text'])
bow_featuresText = bow_vectorizerText.get_feature_names_out()
print(f'Number of unique words in BOW: {len(bow_featuresText)}')
print(f'BOW feature names sample: {bow_featuresText[:10]}')

In [ ]:
bowDataframeText = pd.DataFrame(bow_matrixText.toarray(), columns=bow_featuresText)
bowDataframeText.head()

In [ ]:
bow_vectorizerKeywords = CountVectorizer()
bow_matrixKeywords = bow_vectorizerKeywords.fit_transform(tweet['keyword'])
bow_featuresKeywords = bow_vectorizerKeywords.get_feature_names_out()
print(f'Number of unique words in BOW: {len(bow_featuresKeywords)}')
print(f'BOW feature names sample: {bow_featuresKeywords[:10]}')

In [ ]:
bowDataframeKeywords = pd.DataFrame(bow_matrixKeywords.toarray(), columns=bow_featuresKeywords)
bowDataframeKeywords.head()